In [4]:
#import libraries
import pandas as pd
import numpy as np

#loading the cleaned dataset 
retail_df = pd.read_csv("/Users/manesh/Documents/Manesh /Masters/Portfolios/uk-retail-customer-analytics/Data/online_retail_cleaned.csv")

print("Dataset Shape:", retail_df.shape)

display(retail_df.head())

#preserve datetime types mannually 
retail_df["InvoiceDate"] = pd.to_datetime(
    retail_df["InvoiceDate"]
)

#Verify the input dataset
print("\nDataset Shape: ")
print(retail_df.shape)

print("\nDataset Info: ")
retail_df.info()


print("\nFirst 5 rows of the dataset: ")
display(retail_df.head())


print(
    "Missing CustomerID:",
    retail_df["CustomerID"].isna().sum()
)

#Transaction-Level Feature Engineering
retail_df["TransactionStatus"] = np.where(
    retail_df["InvoiceNo"]
    .astype(str)
    .str.startswith("C"),
    "Cancelled",
    "Completed"
)

retail_df["IsCancellation"] = (
    retail_df["TransactionStatus"] == "Cancelled"
).astype(int)


#Date / Time Features
retail_df["InvoiceYear"] = (
    retail_df["InvoiceDate"].dt.year
)

retail_df["InvoiceMonth"] = (
    retail_df["InvoiceDate"].dt.month
)

retail_df["MonthName"] = (
    retail_df["InvoiceDate"].dt.month_name()
)

retail_df["InvoiceDay"] = (
    retail_df["InvoiceDate"].dt.day
)

retail_df["DayOfWeek"] = (
    retail_df["InvoiceDate"].dt.day_name()
)

retail_df["InvoiceHour"] = (
    retail_df["InvoiceDate"].dt.hour
)

retail_df["InvoiceQuarter"] = (
    retail_df["InvoiceDate"].dt.quarter
)

retail_df["YearMonth"] = (
    retail_df["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

#Revenue Features
retail_df["LineRevenue"] = (
    retail_df["Quantity"]
    * retail_df["UnitPrice"]
)

retail_df["PositiveRevenue"] = np.where(
    retail_df["LineRevenue"] > 0,
    retail_df["LineRevenue"],
    0
)

#Cancellation / Return Features
retail_df["CancellationValue"] = np.where(
    retail_df["IsCancellation"] == 1,
    abs(retail_df["LineRevenue"]),
    0
)

#Product-Level Features
product_features = (
    retail_df
    .groupby(
        ["StockCode", "Description"],
        as_index=False
    )
    .agg(
        TotalQuantity=("Quantity", "sum"),
        TransactionCount=("InvoiceNo", "nunique"),
        UniqueCustomers=("CustomerID", "nunique"),
        TotalRevenue=("LineRevenue", "sum"),
        AverageUnitPrice=("UnitPrice", "mean")
    )
)

#Customer-Level Features
customer_features = (
    retail_df
    .groupby("CustomerID")
    .agg(
        TotalTransactions=(
            "InvoiceNo",
            "nunique"
        ),

        TotalQuantity=(
            "Quantity",
            "sum"
        ),

        TotalRevenue=(
            "LineRevenue",
            "sum"
        ),

        UniqueProducts=(
            "StockCode",
            "nunique"
        ),

        ActiveDays=(
            "InvoiceDate",
            lambda x: x.dt.date.nunique()
        ),

        FirstPurchase=(
            "InvoiceDate",
            "min"
        ),

        LastPurchase=(
            "InvoiceDate",
            "max"
        )
    )
    .reset_index()
)

#RFM Features
reference_date = (
    retail_df["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

sales_df = retail_df[
    (retail_df["TransactionStatus"] == "Completed")
    & (retail_df["Quantity"] > 0)
    & (retail_df["UnitPrice"] > 0)
].copy()

rfm = (
    sales_df
    .groupby("CustomerID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x:
                (reference_date - x.max()).days
        ),

        Frequency=(
            "InvoiceNo",
            "nunique"
        ),

        Monetary=(
            "LineRevenue",
            "sum"
        )
    )
    .reset_index()
)

#Customer Behaviour Features
#Average Order Value
invoice_value = (
    sales_df
    .groupby(
        ["CustomerID", "InvoiceNo"],
        as_index=False
    )
    .agg(
        OrderValue=("LineRevenue", "sum")
    )
)

avg_order_value = (
    invoice_value
    .groupby("CustomerID")["OrderValue"]
    .mean()
    .rename("AverageOrderValue")
)

#Customer lifetime in dataset
customer_features["CustomerLifetimeDays"] = (
    customer_features["LastPurchase"]
    - customer_features["FirstPurchase"]
).dt.days

#Cancellation count
cancellation_features = (
    retail_df[
        retail_df["IsCancellation"] == 1
    ]
    .groupby("CustomerID")
    .agg(
        CancellationCount=(
            "InvoiceNo",
            "nunique"
        ),

        CancellationValue=(
            "CancellationValue",
            "sum"
        )
    )
)

#Customer Segmentation Features
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    4,
    labels=[4, 3, 2, 1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    4,
    labels=[1, 2, 3, 4]
)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

#Validate engineered features
display(
    retail_df[
        [
            "Quantity",
            "UnitPrice",
            "LineRevenue",
            "IsCancellation",
            "CancellationValue"
        ]
    ].describe()
)

display(
    rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ].describe()
)

print(
    customer_features.isnull().sum()
)

print(
    "Negative Recency:",
    (rfm["Recency"] < 0).sum()
)

print(
    "Frequency <= 0:",
    (rfm["Frequency"] <= 0).sum()
)

display(retail_df.head())

display(customer_features.head())

display(rfm.head())

#Save the engineered datasets

#Save the transaction-level engineered data:
retail_df.to_csv(
    "../Data/retail_transaction_features.csv",
    index=False
)

#Customer-level dataset:
customer_features.to_csv(
    "../Data/customer_features.csv",
    index=False
)

#RFM dataset:
rfm.to_csv(
    "../Data/customer_rfm.csv",
    index=False
)

#potentially product-level:
product_features.to_csv(
    "../Data/product_features.csv",
    index=False
)


Dataset Shape: (401604, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionStatus
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Completed
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Completed
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed



Dataset Shape: 
(401604, 9)

Dataset Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401604 entries, 0 to 401603
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   InvoiceNo          401604 non-null  object        
 1   StockCode          401604 non-null  object        
 2   Description        401604 non-null  object        
 3   Quantity           401604 non-null  int64         
 4   InvoiceDate        401604 non-null  datetime64[ns]
 5   UnitPrice          401604 non-null  float64       
 6   CustomerID         401604 non-null  int64         
 7   Country            401604 non-null  object        
 8   TransactionStatus  401604 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(5)
memory usage: 27.6+ MB

First 5 rows of the dataset: 


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionStatus
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Completed
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Completed
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed


Missing CustomerID: 0


...

,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000
mean,92.536422,4.272015,2048.688081
std,100.014169,7.697998,8985.230220
min,1.000000,1.000000,3.750000
25%,18.000000,1.000000,306.482500
50%,51.000000,2.000000,668.570000
75%,142.000000,5.000000,1660.597500
max,374.000000,209.000000,280206.020000


CustomerID              0
TotalTransactions       0
TotalQuantity           0
TotalRevenue            0
UniqueProducts          0
ActiveDays              0
FirstPurchase           0
LastPurchase            0
CustomerLifetimeDays    0
dtype: int64
Negative Recency: 0
Frequency <= 0: 0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionStatus,IsCancellation,...,InvoiceMonth,MonthName,InvoiceDay,DayOfWeek,InvoiceHour,InvoiceQuarter,YearMonth,LineRevenue,PositiveRevenue,CancellationValue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Completed,0,...,12,December,1,Wednesday,8,4,2010-12,15.30,15.30,0.0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed,0,...,12,December,1,Wednesday,8,4,2010-12,20.34,20.34,0.0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Completed,0,...,12,December,1,Wednesday,8,4,2010-12,22.00,22.00,0.0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed,0,...,12,December,1,Wednesday,8,4,2010-12,20.34,20.34,0.0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Completed,0,...,12,December,1,Wednesday,8,4,2010-12,20.34,20.34,0.0


,CustomerID,TotalTransactions,TotalQuantity,TotalRevenue,UniqueProducts,ActiveDays,FirstPurchase,LastPurchase,CustomerLifetimeDays
0,12346,2,0,0.00,1,1,2011-01-18 10:01:00,2011-01-18 10:17:00,0
1,12347,7,2458,4310.00,103,7,2010-12-07 14:57:00,2011-12-07 15:52:00,365
2,12348,4,2341,1797.24,22,4,2010-12-16 19:09:00,2011-09-25 13:13:00,282
3,12349,1,631,1757.55,73,1,2011-11-21 09:51:00,2011-11-21 09:51:00,0
4,12350,1,197,334.40,17,1,2011-02-02 16:01:00,2011-02-02 16:01:00,0


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346,326,1,77183.60,1,1,4,114
1,12347,2,7,4310.00,4,4,4,444
2,12348,75,4,1797.24,2,3,4,234
3,12349,19,1,1757.55,3,1,4,314
4,12350,310,1,334.40,1,1,2,112
